# 57. Word & PowerPoint — `python-docx · python-pptx`
**Python Libraries Lab · Phase 3 · Files & Automation**

_Generate letters, reports and slide decks automatically._

✅ **Runs in Colab** as-is.

python docx creates and edits Word documents, and python pptx does the same for PowerPoint. Instead of copying numbers into a template by hand every week, a script can fill in letters, reports and slide decks from your data in seconds, and produce one per customer or one per city.

**In this notebook:** You will generate a ShopEasy welcome letter for each new seller with a table of their products, read text back from a Word file, then build a four slide monthly review deck with bullet points, a table and a real PowerPoint chart.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q python-docx python-pptx
import importlib.metadata as md
for pkg in ['python-docx', 'python-pptx']:
    print(pkg, md.version(pkg))

## Step 1 · A Word letter with a table
Headings, paragraphs with bold runs, a bullet list and a styled table.

In [ ]:
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

def seller_letter(name, city, products):
    doc = Document()
    doc.add_heading("Welcome to ShopEasy", level=0)
    p = doc.add_paragraph("Dear ")
    p.add_run(name).bold = True
    p.add_run(f", thank you for joining as a seller from {city}. Your first {len(products)} products are now live.")
    doc.add_heading("What happens next", level=2)
    for step in ["We review photos within 24 hours", "Payouts are sent every Friday", "Your dashboard updates daily"]:
        doc.add_paragraph(step, style="List Bullet")
    table = doc.add_table(rows=1, cols=3, style="Light Grid Accent 1")
    for cell, text in zip(table.rows[0].cells, ["Product", "Price", "Stock"]):
        cell.text = text
    for prod, price, stock in products:
        row = table.add_row().cells
        row[0].text, row[1].text, row[2].text = prod, f"Rs. {price:,}", str(stock)
    sign = doc.add_paragraph("\nThe ShopEasy Seller Team"); sign.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    sign.runs[0].font.color.rgb = RGBColor(0x2C, 0x5A, 0x9A); sign.runs[0].font.size = Pt(12)
    path = f"welcome_{name.split()[0].lower()}.docx"
    doc.save(path)
    return path

sellers = [("Kavya Rao", "Mysuru", [("Silk Scarf", 1499, 20), ("Cotton Kurta", 999, 35)]),
           ("Imran Shaikh", "Lucknow", [("Chikan Kurta", 1899, 12)])]
print([seller_letter(*s) for s in sellers])

## Step 2 · Read a Word file
Loop through paragraphs and tables to pull the text back out.

In [ ]:
d = Document("welcome_kavya.docx")
for para in d.paragraphs[:4]:
    print(f"[{para.style.name}] {para.text}")
print([[c.text for c in row.cells] for row in d.tables[0].rows])

## Step 3 · A PowerPoint deck
Each slide uses a layout: 0 is the title slide, 1 is title + content.

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.chart.data import CategoryChartData
from pptx.enum.chart import XL_CHART_TYPE

prs = Presentation()
s = prs.slides.add_slide(prs.slide_layouts[0])
s.shapes.title.text = "ShopEasy · January review"
s.placeholders[1].text = "Sales, cities and next steps"

s = prs.slides.add_slide(prs.slide_layouts[1])
s.shapes.title.text = "Highlights"
body = s.placeholders[1].text_frame
body.text = "Revenue Rs. 22.3 lakh, up 12%"
for line, level in [("Pune is the top city", 0), ("Electronics grew fastest", 0), ("Headphones +40%", 1), ("Return rate fell to 4.1%", 0)]:
    para = body.add_paragraph(); para.text = line; para.level = level

s = prs.slides.add_slide(prs.slide_layouts[5])
s.shapes.title.text = "Revenue by city"
data = CategoryChartData()
data.categories = ["Pune", "Mumbai", "Delhi", "Bengaluru"]
data.add_series("Revenue (Rs. lakh)", (6.6, 5.7, 5.3, 4.6))
chart = s.shapes.add_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, Inches(1), Inches(1.6), Inches(8), Inches(5), data).chart
chart.has_legend = False; chart.plots[0].has_data_labels = True

## Step 4 · A table slide, then save
Tables are shapes too. Save, then download from the Files panel and open in PowerPoint or Google Slides.

In [ ]:
s = prs.slides.add_slide(prs.slide_layouts[5])
s.shapes.title.text = "Top products"
rows = [("Product", "Units", "Revenue"), ("Headphones", "312", "Rs. 7.8 L"), ("Yoga Mat", "290", "Rs. 3.8 L"), ("Steel Bottle", "655", "Rs. 3.3 L")]
tbl = s.shapes.add_table(len(rows), 3, Inches(1), Inches(1.8), Inches(8), Inches(2.4)).table
for r, row in enumerate(rows):
    for c, val in enumerate(row):
        tbl.cell(r, c).text = val
        tbl.cell(r, c).text_frame.paragraphs[0].runs[0].font.size = Pt(18)
prs.save("january_review.pptx")

check = Presentation("january_review.pptx")
for i, slide in enumerate(check.slides, 1):
    print(i, slide.shapes.title.text)

## ✍️ Your turn
Open `january_review.pptx` and store the **number of slides** in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == 4, f"Expected 4, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = len(Presentation("january_review.pptx").slides)

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Document / paragraph / run**
- **Style**
- **Table**
- **Presentation / slide layout**
- **Placeholder**
- **Chart data**

**Next:** 58. Automation & Scheduling (`schedule · pyautogui`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/58_pyautogui_schedule.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install python-docx python-pptx
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.